# Clinically Guided Multimodal Deep Learning for Early Prediction of Functional Deterioration in Knee Osteoarthritis

**Framework:** 3D-HSA-Swin + Clinical Temporal Transformer + Clinically Constrained Cross-Modal Attention Fusion (CC-CAFF)

This notebook provides a patient-level training and evaluation pipeline. It uses MRI volumes from KMAR-50K and requires a separate, genuinely observed longitudinal clinical table for functional-deterioration prediction.

## Dataset compatibility — read before running

KMAR-50K (Mendeley DOI: `10.17632/xw7mrg7ntg.6`) is designed for paired MRI motion-artifact removal. It supplies artifact-corrupted MRI and corresponding reference MRI, plus acquisition metadata. It does **not** supply longitudinal pain, stiffness, mobility, physical-function outcomes, or deterioration labels. Consequently:

- KMAR-50K can supply the MRI modality only.
- A patient-linked clinical CSV with observed repeated visits and a defined future deterioration outcome is mandatory.
- This notebook does not generate proxy clinical labels, infer deterioration from image quality, or fabricate results.
- If the clinical table is absent or incomplete, the notebook stops at validation rather than reporting misleading prediction metrics.

Download KMAR-50K part 1 and part 2 from the Mendeley dataset pages and extract them locally. Confirm the applicable dataset licence and study permissions before use.

## 1. Runtime and dependencies

Recommended: Python 3.10+, PyTorch, CUDA GPU, and sufficient storage/RAM for 3D NIfTI volumes. The model is deliberately compact enough to be adapted to available GPU memory.

In [ ]:
!pip -q install nibabel pandas numpy scikit-learn scipy matplotlib seaborn tqdm

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, re, gc, json, time, math, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("SS-4")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Update these paths after downloading and extracting KMAR-50K part 1 and part 2.
DATA_ROOT = Path("/path/to/KMAR50K_extracted")
CLINICAL_CSV = Path("/path/to/longitudinal_clinical.csv")

# CSV must contain one row per patient visit:
# Patient, visit_month, pain_score, stiffness_score, mobility_score,
# physical_function_score, deterioration_label, future_function_score
# deterioration_label must be a clinically defined future patient-level target,
# consistently repeated for all rows belonging to that patient.
PATIENT_COL = "Patient"
TIME_COL = "visit_month"
CLINICAL_FEATURES = [
    "pain_score", "stiffness_score", "mobility_score", "physical_function_score"
]
TARGET_COL = "deterioration_label"
FUTURE_FUNCTION_COL = "future_function_score"

VOL_SHAPE = (64, 96, 96)       # (D, H, W); reduce for limited GPU memory
MAX_VISITS = 6
BATCH_SIZE = 2
EPOCHS = 30
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PATIENCE = 7
N_SPLITS = 5
AMP = torch.cuda.is_available()

print("Device:", DEVICE)
print("Output directory:", SAVE_DIR.resolve())

## 3. Inspect extracted KMAR-50K files

In [ ]:
assert DATA_ROOT.exists(), f"Set DATA_ROOT to the extracted KMAR-50K directory: {DATA_ROOT}"

nii_files = list(DATA_ROOT.rglob("*.nii")) + list(DATA_ROOT.rglob("*.nii.gz"))
print("NIfTI files found:", len(nii_files))
for p in nii_files[:20]:
    print(p.relative_to(DATA_ROOT))

# KMAR-50K repository folder names commonly distinguish artifact and ground-truth data.
# This project uses the artifact MRI as model input. Do not mix paired scans across splits.
artifact_files = [
    p for p in nii_files
    if "artifact" in str(p).lower() and "testing" not in str(p).lower()
]
if not artifact_files:
    artifact_files = [
        p for p in nii_files
        if "testing_artifact" in str(p).lower()
    ]
print("Candidate artifact MRI volumes:", len(artifact_files))

## 4. Clinical data validation and patient-to-MRI matching

The patient identifier in the clinical table must be reconciled with the anonymized patient identifier encoded in the KMAR filename. The matching function below extracts the leading `year_order_direction_series`-style identifier; inspect the printed matches and adjust `extract_patient_id()` if the local filenames use a different convention.

A single MRI volume is selected per patient for this initial implementation. If multiple sequences exist, the first deterministic match is used; a sequence-selection policy should be specified before final experiments.

In [ ]:
def extract_patient_id(path):
    name = Path(path).name
    # KMAR identifiers are encoded in filenames. Remove NIfTI suffix and processing suffix.
    stem = re.sub(r"\.nii(\.gz)?$", "", name, flags=re.I)
    # Typical patient key: YYYY_Order_Direction; retain series suffix when present.
    m = re.search(r"(\d{4}_\d+_[A-Za-z]+_\d+)", stem)
    if m:
        return m.group(1)
    # Fallback: first four underscore-separated tokens.
    parts = stem.split("_")
    return "_".join(parts[:4]) if len(parts) >= 4 else stem

def build_mri_index(paths):
    index = defaultdict(list)
    for p in paths:
        index[extract_patient_id(p)].append(Path(p))
    return {k: sorted(v, key=lambda x: str(x)) for k, v in index.items()}

mri_index = build_mri_index(artifact_files)
print("Unique MRI patient keys:", len(mri_index))
print("Example keys:", list(mri_index)[:10])

if not CLINICAL_CSV.exists():
    raise FileNotFoundError(
        "Longitudinal clinical CSV not found. KMAR-50K does not contain the required "
        "clinical outcomes. Set CLINICAL_CSV to a real patient-linked clinical table."
    )

clinical = pd.read_csv(CLINICAL_CSV)
required = [PATIENT_COL, TIME_COL, *CLINICAL_FEATURES, TARGET_COL, FUTURE_FUNCTION_COL]
missing = [c for c in required if c not in clinical.columns]
if missing:
    raise ValueError(f"Clinical CSV is missing required columns: {missing}")

clinical[PATIENT_COL] = clinical[PATIENT_COL].astype(str)
clinical[TIME_COL] = pd.to_numeric(clinical[TIME_COL], errors="coerce")
clinical[TARGET_COL] = pd.to_numeric(clinical[TARGET_COL], errors="coerce")
clinical[FUTURE_FUNCTION_COL] = pd.to_numeric(clinical[FUTURE_FUNCTION_COL], errors="coerce")
for c in CLINICAL_FEATURES:
    clinical[c] = pd.to_numeric(clinical[c], errors="coerce")

clinical = clinical.dropna(subset=[PATIENT_COL, TIME_COL, TARGET_COL, FUTURE_FUNCTION_COL]).copy()
if not set(clinical[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError("deterioration_label must be binary 0/1 and clinically defined.")

label_counts = clinical.groupby(PATIENT_COL)[TARGET_COL].nunique()
if (label_counts > 1).any():
    raise ValueError("Each patient must have one consistent patient-level deterioration label.")
future_counts = clinical.groupby(PATIENT_COL)[FUTURE_FUNCTION_COL].nunique()
if (future_counts > 1).any():
    raise ValueError("future_function_score must be one prespecified future outcome per patient.")

clinical = clinical.sort_values([PATIENT_COL, TIME_COL])
clinical = clinical.groupby(PATIENT_COL, group_keys=False).tail(MAX_VISITS)
clinical = clinical.groupby(PATIENT_COL, group_keys=False).apply(
    lambda x: x.sort_values(TIME_COL)
).reset_index(drop=True)

clinical_ids = set(clinical[PATIENT_COL])
matched_ids = clinical_ids.intersection(mri_index)
print("Clinical patients:", len(clinical_ids))
print("MRI patients:", len(mri_index))
print("Matched patients:", len(matched_ids))
print("Clinical patients without MRI:", len(clinical_ids - set(mri_index)))
print("MRI patients without clinical records:", len(set(mri_index) - clinical_ids))

if len(matched_ids) < 10:
    raise ValueError(
        "Too few matched patients. Check filename-to-patient ID mapping and confirm "
        "that clinical Patient IDs use the same anonymized identifiers."
    )

clinical = clinical[clinical[PATIENT_COL].isin(matched_ids)].copy()

## 5. Patient-level split and training-only preprocessing

In [ ]:
patients = np.array(sorted(clinical[PATIENT_COL].unique()))
patient_labels = clinical.groupby(PATIENT_COL)[TARGET_COL].first().reindex(patients).values

# Hold out patients before fitting preprocessing. No patient can appear in more than one split.
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(gss1.split(patients, patient_labels, groups=patients))
trainval_patients, test_patients = patients[trainval_idx], patients[test_idx]

trainval_labels = clinical.groupby(PATIENT_COL)[TARGET_COL].first().reindex(trainval_patients).values
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)
tr_idx, val_idx = next(gss2.split(trainval_patients, trainval_labels, groups=trainval_patients))
train_patients = trainval_patients[tr_idx]
val_patients = trainval_patients[val_idx]

assert not (set(train_patients) & set(val_patients))
assert not (set(train_patients) & set(test_patients))
assert not (set(val_patients) & set(test_patients))

train_rows = clinical[clinical[PATIENT_COL].isin(train_patients)]
clinical_scaler = StandardScaler()
clinical_scaler.fit(train_rows[CLINICAL_FEATURES].fillna(train_rows[CLINICAL_FEATURES].median()))

train_medians = train_rows[CLINICAL_FEATURES].median().fillna(0)
clinical[CLINICAL_FEATURES] = clinical[CLINICAL_FEATURES].fillna(train_medians)
clinical[CLINICAL_FEATURES] = clinical_scaler.transform(clinical[CLINICAL_FEATURES])

print("Patients | train:", len(train_patients), "validation:", len(val_patients), "test:", len(test_patients))
print("Patient-level label counts:")
print(clinical.groupby(PATIENT_COL)[TARGET_COL].first().value_counts().sort_index())

## 6. MRI preprocessing and longitudinal patient dataset

In [ ]:
def load_nifti_volume(path, target_shape=VOL_SHAPE):
    vol = nib.load(str(path)).get_fdata(dtype=np.float32)
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    # Robust intensity normalization within each volume.
    lo, hi = np.percentile(vol, [1, 99])
    vol = np.clip(vol, lo, hi)
    vol = (vol - lo) / (hi - lo + 1e-6)
    factors = [t / s for t, s in zip(target_shape, vol.shape)]
    vol = zoom(vol, factors, order=1)
    vol = np.asarray(vol, dtype=np.float32)
    return vol

class KneeLongitudinalDataset(Dataset):
    def __init__(self, patient_ids, frame, image_index, augment=False):
        self.patient_ids = list(patient_ids)
        self.frame = frame
        self.image_index = image_index
        self.augment = augment
        self.groups = {
            pid: frame[frame[PATIENT_COL] == pid].sort_values(TIME_COL)
            for pid in self.patient_ids
        }

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        rows = self.groups[pid]
        image_path = self.image_index[pid][0]
        volume = load_nifti_volume(image_path)
        if self.augment and random.random() < 0.5:
            volume = volume[:, :, ::-1].copy()
        if self.augment and random.random() < 0.3:
            volume = np.clip(volume * random.uniform(0.9, 1.1), 0, 1)

        x_clin = rows[CLINICAL_FEATURES].to_numpy(dtype=np.float32)
        times = rows[TIME_COL].to_numpy(dtype=np.float32)
        n = min(len(x_clin), MAX_VISITS)
        x_clin = x_clin[-n:]
        times = times[-n:]
        # Normalize time relative to the first available visit.
        times = times - times[0]
        if len(times) > 1:
            times = times / (times.max() + 1e-6)

        clin = np.zeros((MAX_VISITS, len(CLINICAL_FEATURES)), np.float32)
        tarr = np.zeros((MAX_VISITS,), np.float32)
        mask = np.zeros((MAX_VISITS,), np.bool_)
        clin[:n] = x_clin
        tarr[:n] = times
        mask[:n] = True
        label = int(rows[TARGET_COL].iloc[0])

        return {
            "image": torch.from_numpy(volume[None]),
            "clinical": torch.from_numpy(clin),
            "times": torch.from_numpy(tarr),
            "visit_mask": torch.from_numpy(mask),
            "label": torch.tensor(label, dtype=torch.float32),
            "future_function": torch.tensor(float(rows[FUTURE_FUNCTION_COL].iloc[0]), dtype=torch.float32),
            "patient": pid
        }

train_ds = KneeLongitudinalDataset(train_patients, clinical, mri_index, augment=True)
val_ds = KneeLongitudinalDataset(val_patients, clinical, mri_index, augment=False)
test_ds = KneeLongitudinalDataset(test_patients, clinical, mri_index, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
print({k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in batch.items()})

## 7. 3D-HSA-Swin: hierarchical 3D MRI encoder

This implementation uses hierarchical 3D patch embedding, windowed self-attention blocks, and spatial attention gates. It is a compact Swin-inspired implementation intended as a reproducible research starting point, not a pretrained clinical model.

In [ ]:
class SpatialGate3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv3d(channels, max(channels // 4, 8), 1),
            nn.GELU(),
            nn.Conv3d(max(channels // 4, 8), 1, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.gate(x)

class WindowAttention3D(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
    def forward(self, x):
        b, c, d, h, w = x.shape
        # Compact spatial token grid prevents quadratic attention over the full MRI volume.
        pooled = F.adaptive_avg_pool3d(x, (min(d,4), min(h,6), min(w,6)))
        pd, ph, pw = pooled.shape[2:]
        tokens = pooled.flatten(2).transpose(1, 2)
        z = self.norm(tokens)
        a, _ = self.attn(z, z, z, need_weights=False)
        tokens = tokens + a
        tokens = tokens + self.mlp(tokens)
        update = tokens.transpose(1, 2).reshape(b, c, pd, ph, pw)
        update = F.interpolate(update, size=(d,h,w), mode="trilinear", align_corners=False)
        return x + update

class HSAStage3D(nn.Module):
    def __init__(self, in_ch, out_ch, heads=4, downsample=True):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, 3, stride=2 if downsample else 1, padding=1)
        self.norm = nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch)
        self.attn = WindowAttention3D(out_ch, heads)
        self.spatial = SpatialGate3D(out_ch)
    def forward(self, x):
        x = F.gelu(self.norm(self.down(x)))
        # Limit token count by applying attention at the reduced stage resolution.
        x = self.attn(x)
        return self.spatial(x)

class HSA_Swin3D(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, 32, 3, stride=2, padding=1), nn.GroupNorm(8,32), nn.GELU())
        self.stage1 = HSAStage3D(32, 64, heads=4)
        self.stage2 = HSAStage3D(64, 128, heads=4)
        self.stage3 = HSAStage3D(128, 192, heads=4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.proj = nn.Linear(192, out_dim)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.proj(self.pool(x).flatten(1))

## 8. Clinical Temporal Transformer

In [ ]:
class ClinicalTemporalTransformer(nn.Module):
    def __init__(self, n_features, d_model=128, heads=4, layers=2, max_visits=MAX_VISITS):
        super().__init__()
        self.input_proj = nn.Linear(n_features + 1, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_visits, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=4*d_model,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, times, mask):
        z = torch.cat([x, times.unsqueeze(-1)], dim=-1)
        z = self.input_proj(z) + self.pos[:, :z.size(1)]
        z = self.encoder(z, src_key_padding_mask=~mask)
        valid = mask.unsqueeze(-1).float()
        pooled = (z * valid).sum(1) / valid.sum(1).clamp_min(1.0)
        return self.norm(pooled)

## 9. Clinically Constrained Cross-Modal Attention Fusion (CC-CAFF)

In [ ]:
class ClinicallyConstrainedFusion(nn.Module):
    def __init__(self, image_dim=256, clinical_dim=128, fused_dim=256):
        super().__init__()
        self.image_to_clin = nn.MultiheadAttention(image_dim, 4, batch_first=True)
        self.clin_proj = nn.Linear(clinical_dim, image_dim)
        self.cross_gate = nn.Sequential(nn.Linear(image_dim*2, image_dim), nn.Sigmoid())
        self.modality_gate = nn.Sequential(nn.Linear(image_dim*2, 2), nn.Softmax(dim=-1))
        self.out = nn.Sequential(nn.Linear(image_dim*2, fused_dim), nn.LayerNorm(fused_dim), nn.GELU(), nn.Dropout(0.2))
    def forward(self, image_vec, clinical_vec):
        c = self.clin_proj(clinical_vec)
        # One-token cross-modal attention; clinical representation conditions image representation.
        q = image_vec.unsqueeze(1)
        kv = c.unsqueeze(1)
        cross, _ = self.image_to_clin(q, kv, kv, need_weights=False)
        cross = cross.squeeze(1)
        gate = self.cross_gate(torch.cat([image_vec, cross], dim=-1))
        attended_img = image_vec + gate * cross
        weights = self.modality_gate(torch.cat([attended_img, c], dim=-1))
        img_weighted = attended_img * weights[:, 0:1]
        clin_weighted = c * weights[:, 1:2]
        return self.out(torch.cat([img_weighted, clin_weighted], dim=-1)), weights

## 10. Proposed multi-task network

In [ ]:
class ProposedKOAModel(nn.Module):
    def __init__(self, n_clin_features=len(CLINICAL_FEATURES)):
        super().__init__()
        self.image_encoder = HSA_Swin3D(256)
        self.clinical_encoder = ClinicalTemporalTransformer(n_clin_features, 128)
        self.fusion = ClinicallyConstrainedFusion(256, 128, 256)
        self.deterioration = nn.Sequential(nn.Linear(256,128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128,1))
        self.future_function = nn.Linear(256, 1)
        self.group_logits = nn.Linear(256, 2)
    def forward(self, image, clinical, times, visit_mask):
        iv = self.image_encoder(image)
        cv = self.clinical_encoder(clinical, times, visit_mask)
        fused, weights = self.fusion(iv, cv)
        return {
            "deterioration_logit": self.deterioration(fused).squeeze(1),
            "future_function": self.future_function(fused).squeeze(1),
            "group_logits": self.group_logits(fused),
            "fusion_weights": weights
        }

## 11. Ablation variants and comparison architectures

- **3D-HSA-Swin only:** MRI representation only.
- **3D-HSA-Swin + CT-Transformer:** concatenation fusion.
- **3D-HSA-Swin + CT-Transformer + standard cross-attention:** cross-attention without clinical consistency weighting.
- **Proposed:** CC-CAFF with modality-adaptive weighting.

The requested comparison models are EfficientNet, ViT, Bi-LSTM, ResNet, DenseNet, and Proposed. The comparison implementations below are compact 3D/clinical baselines; they share the same patient split and output task. EfficientNet/ResNet/DenseNet names indicate baseline design families rather than pretrained 2D ImageNet weights.

In [ ]:
class Conv3DBackbone(nn.Module):
    def __init__(self, kind="resnet", out_dim=256):
        super().__init__()
        widths = [24, 48, 96, 160]
        blocks = []
        in_c = 1
        for w in widths:
            if kind == "densenet":
                block = nn.Sequential(
                    nn.Conv3d(in_c, w, 3, padding=1), nn.BatchNorm3d(w), nn.GELU(),
                    nn.Conv3d(w, w, 3, padding=1), nn.BatchNorm3d(w), nn.GELU()
                )
            elif kind == "efficientnet":
                block = nn.Sequential(
                    nn.Conv3d(in_c, in_c, 3, padding=1, groups=in_c),
                    nn.Conv3d(in_c, w, 1), nn.BatchNorm3d(w), nn.GELU(),
                    nn.Conv3d(w, w, 3, stride=2, padding=1), nn.BatchNorm3d(w), nn.GELU()
                )
            else:
                block = nn.Sequential(
                    nn.Conv3d(in_c, w, 3, stride=2, padding=1), nn.BatchNorm3d(w), nn.GELU(),
                    nn.Conv3d(w, w, 3, padding=1), nn.BatchNorm3d(w)
                )
            blocks.append(block)
            in_c = w
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(widths[-1], out_dim)
        self.kind = kind
    def forward(self, x):
        for block in self.blocks:
            y = block(x)
            if self.kind == "resnet" and y.shape == x.shape:
                x = F.gelu(x + y)
            elif self.kind == "densenet" and y.shape[1] > x.shape[1]:
                x = torch.cat([y, F.adaptive_avg_pool3d(x, y.shape[2:])], dim=1) if False else y
            else:
                x = y
        return self.fc(self.pool(x).flatten(1))

class ClinicalBiLSTM(nn.Module):
    def __init__(self, n_features, hidden=128):
        super().__init__()
        self.lstm = nn.LSTM(n_features + 1, hidden, batch_first=True, bidirectional=True, num_layers=2, dropout=0.1)
        self.fc = nn.Linear(hidden*2, 256)
    def forward(self, x, times, mask):
        z = torch.cat([x, times.unsqueeze(-1)], dim=-1)
        lengths = mask.sum(1).clamp_min(1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(z, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.fc(torch.cat([h[-2], h[-1]], dim=1))

class ClinicalOnlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = ClinicalTemporalTransformer(len(CLINICAL_FEATURES), 128)
        self.fc = nn.Linear(128, 256)
    def forward(self, x, t, m):
        return self.fc(self.enc(x, t, m))

class BaselineKOAModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.name = name
        if name == "Bi-LSTM":
            self.clin = ClinicalBiLSTM(len(CLINICAL_FEATURES))
            self.img = None
        elif name == "ViT":
            self.img = HSA_Swin3D(256)  # transformer-based 3D image baseline
            self.clin = ClinicalOnlyTransformer()
        elif name in ["EfficientNet", "ResNet", "DenseNet"]:
            self.img = Conv3DBackbone(name.lower(), 256)
            self.clin = ClinicalOnlyTransformer()
        else:
            raise ValueError(name)
        self.head = nn.Sequential(nn.Linear(256 if name == "Bi-LSTM" else 512, 128),
                                  nn.GELU(), nn.Dropout(0.25), nn.Linear(128, 1))
    def forward(self, image, clinical, times, visit_mask):
        if self.name == "Bi-LSTM":
            z = self.clin(clinical, times, visit_mask)
        else:
            z = torch.cat([self.img(image), self.clin(clinical, times, visit_mask)], dim=1)
        return {"deterioration_logit": self.head(z).squeeze(1)}

class AblationKOAModel(nn.Module):
    def __init__(self, variant):
        super().__init__()
        self.variant = variant
        self.img = HSA_Swin3D(256)
        self.clin = ClinicalTemporalTransformer(len(CLINICAL_FEATURES), 128)
        if variant == "image_only":
            dim = 256
        elif variant == "concat":
            dim = 384
        else:
            self.cross = nn.MultiheadAttention(256, 4, batch_first=True)
            dim = 512
        self.head = nn.Sequential(nn.Linear(dim,128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128,1))
    def forward(self, image, clinical, times, visit_mask):
        iv = self.img(image)
        if self.variant == "image_only":
            z = iv
        else:
            cv = self.clin(clinical, times, visit_mask)
            cp = F.pad(cv, (0,128))
            if self.variant == "concat":
                z = torch.cat([iv, cp], dim=1)
            else:
                cross, _ = self.cross(iv.unsqueeze(1), cp.unsqueeze(1), cp.unsqueeze(1), need_weights=False)
                z = torch.cat([iv, cross.squeeze(1)], dim=1)
        return {"deterioration_logit": self.head(z).squeeze(1)}

## 12. Losses, training, and validation

In [ ]:
def compute_loss(outputs, labels, function_targets=None):
    bce = F.binary_cross_entropy_with_logits(outputs["deterioration_logit"], labels)
    loss = bce
    if "future_function" in outputs and function_targets is not None:
        loss = loss + 0.25 * F.smooth_l1_loss(outputs["future_function"], function_targets)
    if "group_logits" in outputs:
        loss = loss + 0.2 * F.cross_entropy(outputs["group_logits"], labels.long())
    return loss

def move_batch(batch):
    return (
        batch["image"].to(DEVICE, non_blocking=True),
        batch["clinical"].to(DEVICE, non_blocking=True),
        batch["times"].to(DEVICE, non_blocking=True),
        batch["visit_mask"].to(DEVICE, non_blocking=True),
        batch["label"].to(DEVICE, non_blocking=True),
        batch["future_function"].to(DEVICE, non_blocking=True)
    )

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    losses = []
    for batch in loader:
        image, clin, times, mask, y, future_y = move_batch(batch)
        out = model(image, clin, times, mask)
        losses.append(compute_loss(out, y, future_y).item())
    return float(np.mean(losses)) if losses else np.nan

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    best_val, best_state, patience = np.inf, None, 0
    history = {"train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        running = []
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            image, clin, times, mask, y, future_y = move_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                out = model(image, clin, times, mask)
                loss = compute_loss(out, y, future_y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            running.append(loss.item())
        tr_loss = float(np.mean(running))
        va_loss = evaluate_loss(model, val_loader)
        scheduler.step(va_loss)
        history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
        print(f"Epoch {epoch+1:03d} | train={tr_loss:.4f} | val={va_loss:.4f}")
        if va_loss < best_val:
            best_val = va_loss
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print("Early stopping.")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

## 13. Classification metrics and inference-time measurement

In [ ]:
@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    ys, probs, elapsed, n = [], [], 0.0, 0
    for batch in loader:
        image, clin, times, mask, y = move_batch(batch)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model(image, clin, times, mask)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        elapsed += time.perf_counter() - start
        p = torch.sigmoid(out["deterioration_logit"])
        ys.extend(y.cpu().numpy().tolist())
        probs.extend(p.cpu().numpy().tolist())
        n += len(y)
    return np.asarray(ys), np.asarray(probs), elapsed / max(n, 1)

def calculate_metrics(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp + 1e-8)
    result = {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1-score": f1_score(y, pred, zero_division=0),
        "Sensitivity": tp / (tp + fn + 1e-8),
        "Specificity": specificity,
        "ROC-AUC": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        "PR-AUC": average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan
    }
    return result

def plot_history(history, title):
    plt.figure(figsize=(7,4))
    plt.plot(history["train_loss"], label="Training")
    plt.plot(history["val_loss"], label="Validation")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title)
    plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

## 14. Train proposed model

In [ ]:
proposed = ProposedKOAModel()
n_params = sum(p.numel() for p in proposed.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

start_train = time.perf_counter()
proposed, proposed_history = train_model(proposed, train_loader, val_loader)
proposed_training_time = time.perf_counter() - start_train
plot_history(proposed_history, "Proposed model training")
torch.save({
    "model_state_dict": proposed.state_dict(),
    "clinical_scaler_mean": clinical_scaler.mean_,
    "clinical_scaler_scale": clinical_scaler.scale_,
    "clinical_features": CLINICAL_FEATURES,
    "volume_shape": VOL_SHAPE
}, SAVE_DIR / "proposed_koa_model.pt")

## 15. Independent test evaluation

In [ ]:
y_test, p_test, proposed_inference_time = predict_model(proposed, test_loader)
proposed_metrics = calculate_metrics(y_test, p_test)
proposed_metrics["Training time (s)"] = proposed_training_time
proposed_metrics["Inference time (s/patient)"] = proposed_inference_time
proposed_results = pd.DataFrame([proposed_metrics], index=["Proposed"])
display(proposed_results)
print(classification_report(y_test, (p_test >= 0.5).astype(int), zero_division=0))

## 16. Comparison: EfficientNet, ViT, Bi-LSTM, ResNet, DenseNet, Proposed

In [ ]:
comparison_rows = []
comparison_models = {}
for name in ["EfficientNet", "ViT", "Bi-LSTM", "ResNet", "DenseNet"]:
    print("\nTraining:", name)
    model = BaselineKOAModel(name)
    t0 = time.perf_counter()
    model, hist = train_model(model, train_loader, val_loader)
    train_seconds = time.perf_counter() - t0
    yt, pt, inf_sec = predict_model(model, test_loader)
    row = calculate_metrics(yt, pt)
    row["Training time (s)"] = train_seconds
    row["Inference time (s/patient)"] = inf_sec
    comparison_rows.append(pd.Series(row, name=name))
    comparison_models[name] = model
    plot_history(hist, f"{name} training")
    torch.save(model.state_dict(), SAVE_DIR / f"{name.lower()}_baseline.pt")

comparison_rows.append(pd.Series(proposed_metrics, name="Proposed"))
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
comparison_table.to_csv(SAVE_DIR / "comparison_metrics.csv")

## 17. Ablation study

In [ ]:
ablation_names = {
    "3D-HSA-Swin only": "image_only",
    "3D-HSA-Swin + CT-Transformer": "concat",
    "3D-HSA-Swin + CT-Transformer + Standard Cross-Attention": "standard_cross",
}
ablation_rows = []
ablation_models = {}
for label, variant in ablation_names.items():
    print("\nAblation:", label)
    model = AblationKOAModel(variant)
    t0 = time.perf_counter()
    model, hist = train_model(model, train_loader, val_loader)
    train_seconds = time.perf_counter() - t0
    yt, pt, inf_sec = predict_model(model, test_loader)
    row = calculate_metrics(yt, pt)
    row["Training time (s)"] = train_seconds
    row["Inference time (s/patient)"] = inf_sec
    ablation_rows.append(pd.Series(row, name=label))
    ablation_models[label] = model
    plot_history(hist, label)
    torch.save(model.state_dict(), SAVE_DIR / f"ablation_{variant}.pt")

ablation_rows.append(pd.Series(proposed_metrics, name="Proposed CC-CAFF"))
ablation_table = pd.DataFrame(ablation_rows)
display(ablation_table)
ablation_table.to_csv(SAVE_DIR / "ablation_metrics.csv")

## 18. ROC and precision-recall curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

plt.figure(figsize=(7,6))
for name, model in comparison_models.items():
    yy, pp, _ = predict_model(model, test_loader)
    if len(np.unique(yy)) == 2:
        fpr, tpr, _ = roc_curve(yy, pp)
        plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yy,pp):.3f})")
fpr, tpr, _ = roc_curve(y_test, p_test)
plt.plot(fpr, tpr, linewidth=2.5, label=f"Proposed (AUC={roc_auc_score(y_test,p_test):.3f})")
plt.plot([0,1],[0,1],"k--",alpha=.5)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Independent-test ROC curves"); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,6))
for name, model in comparison_models.items():
    yy, pp, _ = predict_model(model, test_loader)
    if len(np.unique(yy)) == 2:
        precision, recall, _ = precision_recall_curve(yy, pp)
        plt.plot(recall, precision, label=f"{name} (AP={average_precision_score(yy,pp):.3f})")
precision, recall, _ = precision_recall_curve(y_test, p_test)
plt.plot(recall, precision, linewidth=2.5, label=f"Proposed (AP={average_precision_score(y_test,p_test):.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Independent-test precision-recall curves"); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 19. Save complete experiment results

In [ ]:
comparison_table.to_excel(SAVE_DIR / "comparison_metrics.xlsx")
ablation_table.to_excel(SAVE_DIR / "ablation_metrics.xlsx")
summary = {
    "dataset": "KMAR-50K MRI + externally supplied longitudinal clinical table",
    "patient_split": {
        "train": len(train_patients), "validation": len(val_patients), "test": len(test_patients)
    },
    "clinical_features": CLINICAL_FEATURES,
    "target": TARGET_COL,
    "volume_shape": VOL_SHAPE,
    "seed": SEED,
    "proposed_trainable_parameters": n_params,
    "note": "Metrics are calculated from the supplied clinical labels; no synthetic labels are generated."
}
with open(SAVE_DIR / "experiment_config.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Results saved to:", SAVE_DIR.resolve())

## 20. Required clinical CSV format and study limitations

Example header (values must come from the actual clinical cohort; do not copy placeholder values as data):

```csv
Patient,visit_month,pain_score,stiffness_score,mobility_score,physical_function_score,deterioration_label,future_function_score
```

Each patient must have repeated visits, a stable anonymized identifier that can be matched to the MRI filename, and a clinically justified outcome label. Define the prediction horizon and deterioration criterion prospectively (for example, a validated minimal clinically important difference on a specified functional instrument). The target label and `future_function_score` must represent prespecified future outcomes and must not be derived from information unavailable at prediction time.

### Important methodological requirements before publication

1. Confirm that the MRI cohort and clinical cohort refer to the same patients. KMAR-50K is not an osteoarthritis progression cohort by design.
2. Define the deterioration endpoint, time horizon, baseline prediction time, and handling of censoring/missing visits with clinical collaborators.
3. Ensure MRI sequence selection is prespecified. Multiple scans from one patient must remain within the same split.
4. Use patient-level nested/grouped cross-validation for model selection; retain a genuinely independent test cohort for final reporting.
5. Fit imputation and scaling using training patients only. For strict temporal prediction, ensure every feature is measured before the prediction time.
6. The current notebook implements binary deterioration classification. Future functional-score regression is included in the proposed head, but requires an explicitly defined future score target and corresponding dataset field before its loss can be enabled.
7. Report confidence intervals, calibration, class prevalence, and subgroup performance in addition to discrimination metrics. Small test cohorts can produce unstable ROC-AUC and PR-AUC estimates.

**Do not interpret artifact-removal image pairs as longitudinal functional outcomes.** KMAR-50K is suitable for MRI representation experiments, but the proposed clinical deterioration claim requires a compatible clinical outcome cohort.